# || Feature Engineering : Bureau Dataset ||

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
print("All libraries imported!")
pd.set_option('display.float_format', lambda x: '%.3f' % x)

All libraries imported!


In [2]:
bureau=pd.read_csv('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/raw/bureau.csv')
bureau

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.000,-153.000,NaN,0,91323.000,0.000,NaN,0.000,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.000,NaN,NaN,0,225000.000,171342.000,NaN,0.000,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.000,NaN,NaN,0,464323.500,NaN,NaN,0.000,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.000,NaN,NaN,0.000,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.000,NaN,77674.500,0,2700000.000,NaN,NaN,0.000,Consumer credit,-21,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,259355,5057750,Active,currency 1,-44,0,-30.000,NaN,0.000,0,11250.000,11250.000,0.000,0.000,Microloan,-19,NaN
1716424,100044,5057754,Closed,currency 1,-2648,0,-2433.000,-2493.000,5476.500,0,38130.840,0.000,0.000,0.000,Consumer credit,-2493,NaN
1716425,100044,5057762,Closed,currency 1,-1809,0,-1628.000,-970.000,NaN,0,15570.000,NaN,NaN,0.000,Consumer credit,-967,NaN
1716426,246829,5057770,Closed,currency 1,-1878,0,-1513.000,-1513.000,NaN,0,36000.000,0.000,0.000,0.000,Consumer credit,-1508,NaN


# || Data Cleaning and Processing ||

### Missing Values

In [ ]:
# We know from EDA, these 7 have missing values
missing_col=['AMT_ANNUITY','AMT_CREDIT_MAX_OVERDUE','DAYS_ENDDATE_FACT','AMT_CREDIT_SUM_LIMIT','AMT_CREDIT_SUM_DEBT','DAYS_CREDIT_ENDDATE','AMT_CREDIT_SUM']
# Lets handle them one by one

#### Handling amt_annuity, this is the monthly payment amount that borrower credits in the loan amount.

In [15]:
print(bureau['AMT_ANNUITY'].describe())
print(f"The median : {bureau['AMT_ANNUITY'].median()}")

count      489637.000
mean        15712.758
std        325826.949
min             0.000
25%             0.000
50%             0.000
75%         13500.000
max     118453423.500
Name: AMT_ANNUITY, dtype: float64
The median : 0.0


As seen in the describe also, 50% of the values are 0. And the top 25% values are in range : 13500-118453423.

In [38]:
# For loans that have already ended will have reamining days credit end date in negative or zero.
bureau[['AMT_ANNUITY','AMT_CREDIT_SUM']]
bureau[bureau['AMT_CREDIT_SUM_DEBT']>0][['AMT_CREDIT_SUM','AMT_CREDIT_SUM_DEBT','AMT_ANNUITY','DAYS_CREDIT','DAYS_CREDIT_ENDDATE']]

,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_ANNUITY,DAYS_CREDIT,DAYS_CREDIT_ENDDATE
1,225000.000,171342.000,NaN,-208,1075.000
5,180000.000,71017.380,NaN,-273,27460.000
6,42103.800,42103.800,NaN,-43,79.000
13,89910.000,76905.000,NaN,-96,269.000
17,450000.000,520920.000,NaN,-381,NaN
...,...,...,...,...,...
1716403,1431000.000,1092226.500,NaN,-765,1061.000
1716404,450000.000,432787.500,NaN,-99,997.000
1716412,225000.000,10971.000,NaN,-541,7.000
1716417,67500.000,2466.000,NaN,-740,31128.000


Since 71 % of the value is missing, creating a isMissing column will be useful while keeping the actual values. Another thing is, to get a general idea of a borrower's repayment ability, which is EMI/NMI ratio, the rough annuity can be calculated as (loan paid till now)/(days_credit) * 30.

In [65]:
# imputing credit category wise median into annuity 
category_medians = bureau.groupby('CREDIT_TYPE')['AMT_ANNUITY'].transform('median')
bureau['b_annuity_imp'] = bureau['AMT_ANNUITY'].fillna(category_medians).fillna(0)

# creating expected annuity feature for loans 
debt_paid = bureau['AMT_CREDIT_SUM'] - bureau['AMT_CREDIT_SUM_DEBT']
closed_days = bureau['DAYS_ENDDATE_FACT'] - bureau['DAYS_CREDIT']
active_days = np.abs(bureau['DAYS_CREDIT']) + 1
days_elapsed = np.where(
    bureau['CREDIT_ACTIVE'] == 'Closed', 
    closed_days, 
    active_days
)
bureau['b_exp_annuity']=(debt_paid/days_elapsed)*30.4     # avg of all 12 months

# is missing binary feature 
bureau['b_is_missing'] = bureau['AMT_ANNUITY'].isna().astype('int8')

# repayment intensity feature
bureau['b_annuity_loan_ratio'] = bureau['b_annuity_imp'] / (bureau['AMT_CREDIT_SUM'] + 1)


4 new features : isMissing binary, expected annuity, annuity to loan ratio and imputed annuity were added. Expected annuity gives information about possible EMI of the loan. The imputed annuity is the median annuity of the given credit type. All these can be used to estimate repayment capability. We are keeping the annuity feature with the left over NaN.

In [66]:
bureau[bureau['AMT_ANNUITY']>0][['AMT_ANNUITY','b_annuity_imp','b_exp_annuity','b_is_missing','b_annuity_loan_ratio']]

,AMT_ANNUITY,b_annuity_imp,b_exp_annuity,b_is_missing,b_annuity_loan_ratio
769,2691.000,2691.000,3821.229,0,0.060
781,24462.000,24462.000,4488.919,0,0.202
787,8181.000,8181.000,9318.801,0,0.073
789,8181.000,8181.000,15174.040,0,0.092
790,8061.210,8061.210,0.000,0,8061.210
...,...,...,...,...,...
1716274,43735.500,43735.500,66.287,0,0.076
1716275,43735.500,43735.500,1560.235,0,2.785
1716285,94378.500,94378.500,20962.534,0,0.459
1716290,58554.000,58554.000,37377.049,0,0.065
